# Space-map Tutorial: Xenium, CODEX, and CODEX-Duodenum

This tutorial registers three serial-section datasets end-to-end with the
stable `space_map.api` facade. The data is distributed via a public Google Drive folder. Download them once into
`~/.cache/spacemap-data` (or set `SPACEMAP_DATA_DIR`); `data.fetch` then returns
the local path.

Each dataset is a stack of tissue sections; Space-map aligns them into a
consistent 3D coordinate frame (affine + non-rigid) and reports quality control.

> The default matching method is `auto` (SIFT + the LoFTR deep matcher). LoFTR
> downloads a model checkpoint on first use and is heavier on CPU/GPU. For a
> lighter, checkpoint-free run pass `method="sift_vgg"`.

In [ ]:
# Requires: pip install space-map
import numpy as np
import pandas as pd
from space_map.api import register, RegistrationConfig, data

print('available datasets:', data.available())

## Helper: load a dataset into per-layer arrays

`data.fetch(name)` downloads (and caches) the CSV and returns its local path;
`data.columns(name)` gives the right coordinate/layer column names for that
platform. For very large stacks pass `subsample` to keep the demo fast.

In [ ]:
def load(name, subsample=None, seed=0):
    path = data.fetch(name)
    cols = data.columns(name)
    df = pd.read_csv(path, usecols=[cols['x_col'], cols['y_col'], cols['layer_col']])
    rng = np.random.RandomState(seed)
    def key(v):
        s = str(v)
        if len(s) > 1 and s[0].isalpha() and s[1:].isdigit(): return (0, int(s[1:]))
        try: return (0, int(s))
        except ValueError: return (1, s)
    xys = []
    for k in sorted(df[cols['layer_col']].unique(), key=key):
        sub = df[df[cols['layer_col']] == k]
        if subsample and len(sub) > subsample:
            sub = sub.sample(n=subsample, random_state=rng)
        xys.append(sub[[cols['x_col'], cols['y_col']]].to_numpy(float))
    return xys

def summarize(result):
    pairs = result.qc['internal_consistency']['pairs']
    improved = sum(1 for p in pairs
                   if p.get('chamfer_aligned') is not None
                   and p.get('chamfer_raw') is not None
                   and p['chamfer_aligned'] < p['chamfer_raw'])
    print(f"  layers: {len(result.aligned)}, timings: {result.manifest['timings']}")
    print(f"  QC: {improved}/{len(pairs)} adjacent pairs improved; "
          f"all finite = {result.qc['finiteness']['all_finite']}")

## 1. Xenium (polyp, transcriptomics)

20 sections numbered 1–20. Coordinates in `x`/`y`.

In [ ]:
xys = load('xenium_polyp', subsample=20000)
res = register(xys, RegistrationConfig(workdir='out/xenium', seed=0, method='auto'))
res.save('out/xenium')
summarize(res)

## 2. CODEX (colon, proteomics)

16 sections labelled `S1`–`S16`, grouped by the `array` column.

In [ ]:
xys = load('codex_colon', subsample=20000)
res = register(xys, RegistrationConfig(workdir='out/codex', seed=0, method='auto'))
res.save('out/codex')
summarize(res)

## 3. CODEX-Duodenum

One file with both the registration input (`raw_x`/`raw_y`) and a reference
alignment (`x`/`y`) per cell. We register the raw coordinates; the `x`/`y`
columns give a reference alignment you can compare against.

In [ ]:
xys = load('codex_duodenum', subsample=20000)
res = register(xys, RegistrationConfig(workdir='out/duodenum', seed=0, method='auto'))
res.save('out/duodenum')
summarize(res)

## Outputs

Each `out/<dataset>/` directory contains `aligned/<layer>.npy` plus
`qc.json`, `manifest.json`, `transforms.json`, and `warnings.json`.

Drop `subsample=` to run on the full stacks (Xenium/CODEX use a lot of memory
at full resolution). See `examples/run_platforms.py` for the command-line
equivalent.